# Ops Copilot: RAG walkthrough (Phase 1)

This notebook walks the pipeline end to end on the 10 simulated Meridian Supply Co. documents:

docs -> cleaning -> chunks -> embeddings -> BM25 vs dense vs hybrid -> grounded answer with citations -> refusal -> eval summary.

It needs a local Ollama with `nomic-embed-text` and `llama3.1:8b`. Settings are set in the first cell via `os.environ`; no `.env` file is read.

## 0. Setup
Select the local Ollama provider and move to the repository root so paths are relative.

In [1]:
import os
from pathlib import Path

os.environ["LLM_PROVIDER"] = "ollama"
os.environ["OLLAMA_MODEL"] = "llama3.1:8b"

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print("working dir: <repo root> | docs:", len(list(Path("data/docs").glob("*.md"))))

working dir: <repo root> | docs: 10


## 1. Documents and cleaning (Week 1)
Raw SOPs carry front matter, a CONFIDENTIAL banner, template comments and a document-control footer. None of it helps retrieval and some of it
hurts (every chunk would match "Document control"). `clean()` strips it and keeps headings, which the chunker needs.

In [2]:
from ops_copilot.ingest import clean, load_docs

raw = Path("data/docs/WH-001.md").read_text(encoding="utf-8")
cleaned = clean(raw)


def noisy(text):
    keys = ("CONFIDENTIAL", "<!--", "Document control", "Uncontrolled")
    return [ln.strip() for ln in text.splitlines() if any(k in ln for k in keys)]


print("BEFORE (banner/footer lines in raw WH-001.md):")
for ln in noisy(raw):
    print("  ", ln)
print("\nAFTER clean(): noisy lines left =", len(noisy(cleaned)))
print("chars raw -> cleaned:", len(raw), "->", len(cleaned))
print("\nFirst lines after cleaning:")
print(cleaned[:200])

docs = load_docs("data/docs")
print([d.doc_id for d in docs])

BEFORE (banner/footer lines in raw WH-001.md):
   CONFIDENTIAL — Meridian Supply Co. internal use only
   <!-- template: SOP-TPL-04 rev C; maintained by Document Control; do not edit outside the controlled repository -->
   Document control: WH-001 v2.1 | Meridian Supply Co. Document Control | Page 1 of 1 | Uncontrolled when printed

AFTER clean(): noisy lines left = 0
chars raw -> cleaned: 4607 -> 4167

First lines after cleaning:
# WH-001 Receiving & Putaway SOP

## 1. Purpose

### 1.1 Objective
This procedure defines how inbound freight is checked in, verified, counted, inspected and put away at the Meridian Supply Co. distri
['CS-001', 'FIN-001', 'PR-001', 'PR-002', 'QA-001', 'WH-001', 'WH-002', 'WH-003', 'WH-004', 'WH-005']


## 2. Chunking (Week 1)
Header-aware: split on `#`/`##`/`###`, then by size (~1600 chars, about 400 tokens, 200 overlap). Each chunk gets a context prefix
`DOC title > section` so it still makes sense on its own, and an id `DOC#section#n`, which is what citations point at.

In [3]:
from collections import Counter

from ops_copilot.chunk import chunk_docs

chunks = chunk_docs(docs)
print("total chunks:", len(chunks))
for doc_id, n in sorted(Counter(c.doc_id for c in chunks).items()):
    print(f"  {doc_id}: {n}")

buckets = Counter()
for c in chunks:
    buckets[min(len(c.text) // 200 * 200, 1400)] += 1
print("\nchunk size (chars) histogram (one # = 3 chunks):")
for b in sorted(buckets):
    label = f"{b}-{b + 199}" if b < 1400 else "1400+"
    print(f"  {label:>10} | {'#' * (buckets[b] // 3)} {buckets[b]}")

s = chunks[3]
print("\nmetadata sample:")
print(
    {
        "id": s.id,
        "doc_id": s.doc_id,
        "section": s.section,
        "section_title": s.section_title,
        "chars": len(s.text),
    }
)
print(s.text[:300])

total chunks: 163
  CS-001: 15
  FIN-001: 17
  PR-001: 13
  PR-002: 16
  QA-001: 17
  WH-001: 17
  WH-002: 19
  WH-003: 16
  WH-004: 17
  WH-005: 16

chunk size (chars) histogram (one # = 3 chunks):
       0-199 | ################# 53
     200-399 | ################################## 103
     400-599 | ## 7

metadata sample:
{'id': 'CS-001#3.1#0', 'doc_id': 'CS-001', 'section': '3.1', 'section_title': 'Order cut-off for same-day shipping', 'chars': 285}
CS-001 Key-Account Service Levels > 3.1 Order cut-off for same-day shipping
Orders received by 3 pm local time on a working day ship the same day. Orders received after 3 pm ship the next working day. The cut-off is applied by order-entry timestamp and is shown on the customer portal.


## 3. Embeddings (Week 2)
`nomic-embed-text` (via Ollama) maps text to a vector. Similar meaning gives a higher cosine similarity. The Chroma collection is built on these vectors.

In [4]:
import math

from ops_copilot.config import Settings
from ops_copilot.llm import get_chat_model, get_embeddings

settings = Settings.from_env()
emb = get_embeddings(settings)

a, b, c = emb.embed_documents(
    [
        "How often are A-class items cycle counted?",
        "A items are counted monthly as part of the cycle count schedule.",
        "Forklift operators must hold a current certification.",
    ]
)


def cosine(u, v):
    dot = sum(x * y for x, y in zip(u, v, strict=True))
    return dot / (math.sqrt(sum(x * x for x in u)) * math.sqrt(sum(y * y for y in v)))


print("vector dim:", len(a), "| first 3 values:", [round(x, 3) for x in a[:3]])
print("cosine(question, cycle-count sentence):", round(cosine(a, b), 3))
print("cosine(question, forklift sentence):   ", round(cosine(a, c), 3))

vector dim: 768 | first 3 values: [-0.045, -0.005, -0.187]
cosine(question, cycle-count sentence): 0.75
cosine(question, forklift sentence):    0.409


## 4. BM25 vs dense vs hybrid (Week 2)
BM25 matches exact words; dense matches meaning; hybrid fuses the two rankings with reciprocal rank fusion,
`score = sum 1 / (60 + rank)`. The same question goes through all three; rank is `-` when a chunk is outside that retriever's top 10.

In [5]:
from ops_copilot.index import Index
from ops_copilot.retrieve import retrieve

index = Index(chunks, emb, "data/cache/notebook-chroma", settings.embed_model)
Q = "How often are A-class items cycle counted?"

for mode in ("bm25", "dense", "hybrid"):
    r = retrieve(index, Q, mode, settings.top_k, settings.rrf_k, settings.candidates)
    print(f"== {mode} (confidence = max dense cosine = {r.confidence:.3f})")
    print(f"{'#':>2} {'chunk':<16} {'bm25':>5} {'dense':>6} {'cos':>6} {'rrf':>7}")
    for i, h in enumerate(r.hits, 1):
        print(
            f"{i:>2} {h.chunk.id:<16} {h.bm25_rank or '-':>5} {h.dense_rank or '-':>6} "
            f"{h.dense_score:>6.3f} {h.rrf:>7.4f}"
        )

== bm25 (confidence = max dense cosine = 0.738)
 # chunk             bm25  dense    cos     rrf
 1 WH-002#2.2#0         1      1  0.738  0.0328
 2 WH-002#3.3#0         2      5  0.627  0.0315
 3 WH-002#1.1#0         3      7  0.617  0.0308
 4 WH-002#2.1#0         4      2  0.708  0.0318
== dense (confidence = max dense cosine = 0.738)
 # chunk             bm25  dense    cos     rrf
 1 WH-002#2.2#0         1      1  0.738  0.0328
 2 WH-002#2.1#0         4      2  0.708  0.0318
 3 WH-002#2.3#0         -      3  0.647  0.0159
 4 WH-002#5.2#0         -      4  0.629  0.0156
== hybrid (confidence = max dense cosine = 0.738)
 # chunk             bm25  dense    cos     rrf
 1 WH-002#2.2#0         1      1  0.738  0.0328
 2 WH-002#2.1#0         4      2  0.708  0.0318
 3 WH-002#3.3#0         2      5  0.627  0.0315
 4 WH-002#1.1#0         3      7  0.617  0.0308


## 5. Grounded answer with citations
Hybrid top-4 chunks go into a prompt that allows only the given context and requires `[DOC §section]` citations.
A validator then checks every citation against the retrieved chunks; an invalid citation turns the answer into a refusal.

In [6]:
from ops_copilot.pipeline import Pipeline

pipe = Pipeline(settings, emb, get_chat_model(settings))


def show(res):
    print(
        "refused:",
        res["refused"],
        "| reason:",
        res["refusal_reason"],
        "| confidence:",
        round(res["confidence"], 3),
    )
    print("answer:", res["answer"])
    print("citations:", [(c["doc_id"], c["section"], c["valid"]) for c in res["citations"]])
    print("timings ms:", {k: round(v, 1) for k, v in res["timings_ms"].items()})


show(pipe.ask(Q))

refused: False | reason: None | confidence: 0.738
answer: A-class items are counted twelve times a year [WH-002 §2.2].
citations: [('WH-002', '2.2', True)]
timings ms: {'embed': 9.5, 'bm25': 0.2, 'dense': 2.2, 'rrf': 0.0, 'gate': 0.0, 'llm': 8894.4}


## 6. Refusal (the confidence gate)
Q14 asks about forklift recertification, which is not in the corpus. Retrieval confidence falls below the calibrated threshold (0.61),
so the gate refuses **before the LLM is called**: the `llm` timing is 0 ms.

In [7]:
res = pipe.ask("How often must forklift operators be recertified?")
show(res)
assert res["refused"] and res["timings_ms"]["llm"] == 0.0

refused: True | reason: below_gate | confidence: 0.548
answer: I couldn't find this in Meridian's documents.
citations: []
timings ms: {'embed': 792.2, 'bm25': 0.9, 'dense': 4.7, 'rrf': 0.1, 'gate': 0.0, 'llm': 0.0}


## 7. Eval summary
The latest 15-question eval (`eval/latest.json`, details in `eval/report.md`). Two of the five KPIs **fail** their pass bars:
faithfulness (a same-model judge, 8B generator) and p95 latency (local 8B model). This is the honest Phase 1 result; nothing was loosened to pass.

In [8]:
import json

ev = json.loads(Path("eval/latest.json").read_text())
m, bars = ev["metrics"], ev["pass_bars"]
rows = [
    (
        "Faithfulness",
        f"{m['faithfulness']:.3f}",
        f">= {bars['faithfulness']}",
        m["faithfulness"] >= bars["faithfulness"],
    ),
    ("hit@4", f"{m['hit_at_4']:.3f}", f">= {bars['hit_at_4']}", m["hit_at_4"] >= bars["hit_at_4"]),
    ("MRR", f"{m['mrr']:.3f}", f">= {bars['mrr']}", m["mrr"] >= bars["mrr"]),
    (
        "Refusal correct",
        f"{m['refusal_correct']}/{m['refusal_total']}",
        f"{bars['refusal_correct']}/{m['refusal_total']}",
        m["refusal_correct"] >= bars["refusal_correct"],
    ),
    (
        "p95 latency",
        f"{m['p95_latency_ms']:,.0f} ms",
        f"< {bars['p95_latency_ms']:,} ms",
        m["p95_latency_ms"] < bars["p95_latency_ms"],
    ),
]
print(
    f"run: {ev['run_at']} | model: {ev['model']} | embeddings: {ev['embed_model']} "
    f"| gate: {ev['gate_threshold']}\n"
)
print(f"{'KPI':<16} {'result':>12} {'bar':>12}  status")
for name, val, bar, ok in rows:
    print(f"{name:<16} {val:>12} {bar:>12}  {'PASS' if ok else 'FAIL'}")

print(f"\n{'Q':<4} {'type':<13} {'refused':<20} {'hit':<4} {'faith':>5} {'ms':>7}  passed")
for q in ev["questions"]:
    refused = ("yes (" + q["refusal_reason"] + ")") if q["refused"] else "no"
    hit = "-" if q["hit"] is None else ("yes" if q["hit"] else "no")
    f = "-" if q["faithfulness"] is None else f"{q['faithfulness']:.2f}"
    print(
        f"{q['id']:<4} {q['type']:<13} {refused:<20} {hit:<4} {f:>5} {q['latency_ms']:>7,.0f}  "
        f"{'yes' if q['passed'] else 'no'}"
    )
print("passed:", sum(q["passed"] for q in ev["questions"]), "/", len(ev["questions"]))

run: 2026-11-20T09:57:08+00:00 | model: llama3.1:8b | embeddings: nomic-embed-text | gate: 0.61

KPI                    result          bar  status
Faithfulness            0.381       >= 0.9  FAIL
hit@4                   1.000      >= 0.85  PASS
MRR                     1.000       >= 0.7  PASS
Refusal correct           2/2          2/2  PASS
p95 latency         13,808 ms   < 8,000 ms  FAIL

Q    type          refused              hit  faith      ms  passed
Q1   direct        no                   yes   0.00   3,362  no
Q2   direct        no                   yes   1.00   3,650  yes
Q3   direct        no                   yes   0.00   3,798  no
Q4   direct        no                   yes   1.00   3,922  yes
Q5   direct        no                   yes   1.00   4,211  yes
Q6   direct        no                   yes   1.00   4,427  yes
Q7   direct        no                   yes   0.50   6,545  no
Q8   direct        no                   yes   1.00   4,418  yes
Q9   multi         no         